# Classificador de fraude por rastreador

Usa os três modelos treinados no notebook anterior (resultados/modelos_finais.pkl) para classificar o arquivo de auditoria de um rastreador como **FRAUDE** ou **NÃO FRAUDE**, com o risco e a confiança de cada modelo. Nenhum indício isolado declara fraude: um episódio só é fraude se reunir pelo menos dois tipos de indício ao mesmo tempo e o risco combinado passar de 50%.

Seções: análise de um arquivo, ranking de uma pasta e interface web.

## Preparação do ambiente

In [ ]:
import os, sys

REPO = "https://github.com/Dapalha/projeto-monografia-regressao.git"
PASTA_REPO = "/content/projeto-monografia-regressao"

if "google.colab" in sys.modules:
    if not os.path.exists(PASTA_REPO):
        !git clone -q {REPO} {PASTA_REPO}
    %cd {PASTA_REPO}
    !pip -q install -r requirements.txt
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.getcwd())

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from fraude import figuras as fig
from fraude.analise import analisar_arquivo, grafico_risco, tabela_episodios, tabela_resumo

DRIVE = Path("/content/drive/MyDrive")
DIR_BRUTOS = DRIVE / "dados_brutos"          # planilhas de auditoria originais
DIR_TRATADOS = DRIVE / "dataset_tratado"     # *_treated.csv
DIR_RESULT = DIR_TRATADOS / "resultados"
DIR_FIG = DIR_RESULT / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)

pacote = joblib.load(DIR_RESULT / "modelos_finais.pkl")
fig.estilo()
print("modelos:", ", ".join(pacote["modelos"]))

## 1. Análise de um arquivo

In [ ]:
ARQUIVO = DIR_BRUTOS / "69251992613cd7d158638cb9_auditoria.csv"   # troque pelo arquivo desejado

info, resultados, janelas = analisar_arquivo(ARQUIVO, pacote)
print(f"{info['arquivo']} | {info['periodo']} | consenso: {info['consenso']} ({info['votos_fraude']} modelos)")
display(tabela_resumo(resultados))
grafico_risco(janelas, resultados, pacote["decisao"], caminho=DIR_FIG / f"risco_{Path(ARQUIVO).stem}.png")
plt.show()
tabela_episodios(resultados, so_fraude=True).head(20)

### Enviar um arquivo do computador

In [ ]:
from google.colab import files

for nome in files.upload():
    info, resultados, janelas = analisar_arquivo(nome, pacote)
    print(f"{info['arquivo']}: {info['consenso']} ({info['votos_fraude']} modelos)")
    display(tabela_resumo(resultados))
    grafico_risco(janelas, resultados, pacote["decisao"])
    plt.show()

## 2. Ranking de uma pasta

In [ ]:
linhas = []
for fp in sorted(DIR_BRUTOS.glob("*_auditoria.csv")):
    try:
        info, resultados, _ = analisar_arquivo(fp, pacote)
    except Exception as e:
        print(f"{fp.name}: {type(e).__name__}: {e}")
        continue
    linha = {"arquivo": info["arquivo"], "periodo": info["periodo"], "consenso": info["consenso"]}
    for nome, r in resultados.items():
        linha[f"{nome} - veredito"] = r["veredito"]
        linha[f"{nome} - risco (%)"] = r["risco_pct"]
    linhas.append(linha)

ranking = pd.DataFrame(linhas)
riscos = [c for c in ranking.columns if c.endswith("risco (%)")]
ranking["risco medio (%)"] = ranking[riscos].mean(axis=1).round(1)
ranking = ranking.sort_values("risco medio (%)", ascending=False)
ranking.to_csv(DIR_RESULT / "ranking_rastreadores.csv", sep=";", index=False)
print(ranking["consenso"].value_counts())
ranking.head(30)

## 3. Interface web
Gera uma página para enviar o arquivo e receber o resultado. O link público vale por 72 horas; para uso permanente, rode o app.py deste repositório.

In [ ]:
from fraude.interface import criar_interface

criar_interface(pacote).launch(share=True)